## 1. Persiapan Lingkungan dan Google Drive
Cell ini bertugas untuk menghubungkan Google Colab dengan Google Drive Anda, sehingga kita bisa membaca dataset berformat ZIP dan model hasil training sebelumnya.


In [ ]:
from google.colab import drive
import os
import shutil
import zipfile
drive.mount('/content/drive')

## 2. Instalasi Dependensi
Menginstal pustaka Ultralytics (YOLO) dan utilitas lainnya. Versi Pillow dikunci untuk mencegah crash pada C-extension yang sering terjadi di environment Colab terbaru.


In [ ]:
import IPython
import PIL
pil_ver = PIL.__version__
print(f"Mengunci versi Pillow ke {pil_ver} untuk mencegah crash C-extension...")
IPython.get_ipython().system(f"pip install -q ultralytics Pillow=={pil_ver}")

import importlib
import site
importlib.reload(site)
importlib.invalidate_caches()

print("Environment siap!")

## 3. Pengaturan Path Direktori
Mendefinisikan lokasi penyimpanan dataset di storage lokal Colab (`/content/vnetra_master_dataset`) serta letak model-model yang telah ditraining di Google Drive.


In [ ]:
EXPERIMENT_ID = 1 # Ganti sesuai ID eksperimen Anda
DRIVE_BASE_DIR = f'/content/drive/MyDrive/YOLO/eksperimen_{EXPERIMENT_ID}'
OUTPUT_DIR = f'{DRIVE_BASE_DIR}/output'
INPUT_DIR = f'{DRIVE_BASE_DIR}/input'

MASTER_DIR = '/content/vnetra_master_dataset'
DATA_YAML = f'{MASTER_DIR}/data.yaml'

# Path model yang sudah di-training
MODEL_PT = f'{OUTPUT_DIR}/best.pt'
MODEL_FP32 = f'{OUTPUT_DIR}/best_fp32.tflite'

print('=== SETUP PATH SELESAI ===')

## 4. Ekstraksi Dataset untuk Kalibrasi
Kuantisasi INT8 WAJIB membutuhkan sekumpulan data latih/validasi agar framework LiteRT dapat melakukan *Calibration* (penentuan rentang nilai aktivasi untuk setiap layer). Dataset diakses dari format ZIP di GDrive dan diekstrak ke lokal.


In [ ]:
import os
import shutil
import zipfile
import yaml

# PATH ZIP DATASET DI GDRIVE (SESUAIKAN JIKA PERLU)
DATASET_ZIP_PATH = f'{INPUT_DIR}/vnetra_master_dataset.zip'

master_dir = '/content/vnetra_master_dataset'
data_yaml = f'{master_dir}/data.yaml'

if os.path.exists(master_dir):
    shutil.rmtree(master_dir)
os.makedirs(master_dir, exist_ok=True)

print(f"Mengekstrak {DATASET_ZIP_PATH} ke {master_dir}...")
if os.path.exists(DATASET_ZIP_PATH):
    with zipfile.ZipFile(DATASET_ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall('/content/')
    print("Ekstraksi Selesai!")
else:
    print("[ERROR] File ZIP dataset tidak ditemukan di Drive! INT8 butuh dataset untuk kalibrasi.")

## 4.5 Verifikasi Distribusi Dataset Kalibrasi
Menampilkan ringkasan tabel distribusi instance dari dataset yang baru diekstrak untuk memastikan dataset (Train, Valid, Test) utuh dan valid digunakan sebagai data representatif kalibrasi INT8.

In [ ]:
import pandas as pd
import os

print('=== MEMBACA DATA.YAML ===')
if os.path.exists(data_yaml):
    with open(data_yaml, 'r') as f:
        yaml_data = yaml.safe_load(f)
        if isinstance(yaml_data.get('names'), dict):
            master_classes = [yaml_data['names'][i] for i in range(len(yaml_data['names']))]
        else:
            master_classes = yaml_data.get('names', [])
else:
    master_classes = []
    print('[ERROR] data.yaml tidak ditemukan!')

def count_images(directory):
    if not os.path.exists(directory): return 0
    return len([f for f in os.listdir(directory) if f.endswith(('.jpg', '.jpeg', '.png'))])

train_count = count_images(f'{master_dir}/train/images')
valid_count = count_images(f'{master_dir}/valid/images')
test_count  = count_images(f'{master_dir}/test/images')
total_images = train_count + valid_count + test_count

print("=== Statistik Keseluruhan ===")
print(f"Total Lembar Gambar (All) : {total_images} gambar")
print(f"Total Gambar Training     : {train_count} gambar")
print(f"Total Gambar Validasi     : {valid_count} gambar")
print(f"Total Gambar Testing      : {test_count} gambar")
print("=============================")
print("")

def count_instances_per_class(label_dir, num_classes):
    counts = {i: 0 for i in range(num_classes)}
    if not os.path.exists(label_dir): return counts
    for lbl_file in os.listdir(label_dir):
        if not lbl_file.endswith('.txt'): continue
        with open(os.path.join(label_dir, lbl_file), 'r') as f:
            for line in f:
                parts = line.strip().split()
                if parts: counts[int(parts[0])] += 1
    return counts

train_cls = count_instances_per_class(f'{master_dir}/train/labels', len(master_classes))
valid_cls = count_instances_per_class(f'{master_dir}/valid/labels', len(master_classes))
test_cls  = count_instances_per_class(f'{master_dir}/test/labels', len(master_classes))

data_report = []
total_train = 0
total_valid = 0
total_test = 0
global_total = 0

for i, cls_name in enumerate(master_classes):
    t_train = train_cls[i]
    t_valid = valid_cls[i]
    t_test = test_cls[i]
    t_total = t_train + t_valid + t_test

    total_train += t_train
    total_valid += t_valid
    total_test += t_test
    global_total += t_total

    data_report.append({
        'ID': i,
        'Kelas': cls_name,
        'Train (Inst)': t_train,
        'Valid (Inst)': t_valid,
        'Test (Inst)': t_test,
        'Total Instance': t_total
    })

data_report.append({
    'ID': '-',
    'Kelas': 'TOTAL KESELURUHAN',
    'Train (Inst)': total_train,
    'Valid (Inst)': total_valid,
    'Test (Inst)': total_test,
    'Total Instance': global_total
})

df_report = pd.DataFrame(data_report)
display(df_report)


## 5. Evaluasi Model Original PyTorch (.pt)
Kita memanggil model asli sebelum dikonversi dan melakukan uji pada dataset test untuk mendapatkan nilai mAP50 asli sebagai *baseline* (patokan utama).


In [ ]:
from ultralytics import YOLO

print('\n=== MEMANGGIL MODEL ASLI (.pt) ===')
model = YOLO(MODEL_PT)

map_pt = 0.0
try:
    val_pt = model.val(data=DATA_YAML, split='test')
    map_pt = val_pt.box.map50
    print(f'\n[INFO] mAP@50 Model Asli (PT): {map_pt:.4f}')
except Exception as e:
    print(f'Evaluasi model asli gagal: {e}')

## 6. Evaluasi Model FP32 LiteRT (.tflite)
Kita memanggil model FP32 (jika sudah di-export pada saat proses training selesai) untuk melihat apakah ada degradasi presisi akibat konversi standar dari format ONNX ke TFLite.


In [ ]:
print('\n=== MEMANGGIL MODEL FP32 (.tflite) ===')
map_fp32 = 0.0
if os.path.exists(MODEL_FP32):
    model_fp32 = YOLO(MODEL_FP32, task='detect')
    try:
        val_fp32 = model_fp32.val(data=DATA_YAML, split='test')
        map_fp32 = val_fp32.box.map50
        print(f'\n[INFO] mAP@50 Model FP32: {map_fp32:.4f}')
    except Exception as e:
        print(f'Evaluasi model FP32 gagal: {e}')
else:
    print(f'[WARNING] Model FP32 tidak ditemukan di {MODEL_FP32}')

## 7. Proses Kuantisasi INT8 LiteRT
Tahap utama. Kita mengekspor model asli `.pt` ke format LiteRT dengan `int8=True`. Pada tahap ini YOLO akan menggunakan *Representative Dataset* (berasal dari `DATA_YAML`) untuk menekan ukuran model hingga 4x lebih kecil.


In [ ]:
print('\n=== MEMULAI KUANTISASI INT8 LITERT ===')
print('Mengekspor model ke format INT8. Ini mungkin akan memakan waktu 5-15 menit karena kalibrasi...')

export_int8 = model.export(
    format='litert',
    int8=True,
    data=DATA_YAML,
    optimize=True
)

print('===========================================================')
print('Export INT8 Selesai! Lokasi file TFLite:', export_int8)
print('===========================================================')

## 8. Evaluasi Model INT8 LiteRT (.tflite)
Setelah model berhasil dikuantisasi ke 8-bit, kita uji lagi pada dataset test. Secara teori ukuran file akan jauh lebih kecil dan inferensi lebih cepat, namun mAP50 akan sedikit turun. Kita perlu mengukur seberapa jauh penurunannya.


In [ ]:
print('\n=== MEMANGGIL MODEL INT8 (.tflite) ===')
model_int8 = YOLO(export_int8, task='detect')
map_int8 = 0.0

try:
    val_int8 = model_int8.val(data=DATA_YAML, split='test')
    map_int8 = val_int8.box.map50
    print(f'\n[INFO] mAP@50 Model INT8: {map_int8:.4f}')
except Exception as e:
    print(f'Evaluasi model INT8 gagal: {e}')

## 9. Simpan Model INT8 ke Google Drive
Menyimpan secara permanen file `.tflite` versi INT8 ini ke dalam folder output eksperimen Anda di Google Drive.


In [ ]:
print('\n=== MENYIMPAN MODEL INT8 KE GOOGLE DRIVE ===')
dest_path = os.path.join(OUTPUT_DIR, 'best_int8.tflite')
if os.path.exists(export_int8):
    shutil.copy(export_int8, dest_path)
    print(f'Model INT8 berhasil dicadangkan ke {dest_path}')
else:
    print('[ERROR] File INT8 hasil export tidak ditemukan!')

## 10. Visualisasi Perbandingan Performa
Grafik batang (Bar Chart) untuk membandingkan secara visual Akurasi (mAP50) dari Model Asli, Model FP32, dan Model INT8 hasil kompresi.


In [ ]:
import matplotlib.pyplot as plt

labels = ['Original (.pt)', 'LiteRT (FP32)', 'LiteRT (INT8)']
values = [map_pt, map_fp32, map_int8]

plt.figure(figsize=(8, 6))
bars = plt.bar(labels, values, color=['#1f77b4', '#ff7f0e', '#2ca02c'])

plt.title('Perbandingan mAP@50: Original vs FP32 vs INT8', fontsize=14)
plt.ylabel('mAP@50', fontsize=12)
plt.ylim(0, max(values) * 1.2 if max(values) > 0 else 1.0)

# Tambahkan teks label di atas setiap bar
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 0.01, f'{yval:.4f}', ha='center', va='bottom', fontsize=11)

plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()